# Audit: new TS (pn validity mask) vs previous TS vs TI, 1990-2020

Repeats every check from the TS/TI investigation for the new TS version, next to the previous version
(`5_Typology/TS_python_only_v2`, before the pn mask).

Run the cells top to bottom. **Pass A** and **Pass B** each read the rasters once, block by block. On the full
data expect roughly 30-90 minutes in total; both passes print progress.

Outputs (also printed): CSV tables and two maps in `data/5_Typology/audit_pnmask/`.

| Section | Question it answers |
|---|---|
| 0 | Do all files exist, line up, and were the steps run in order? |
| A1 | Footprint and agreement, previous vs new |
| A2 | Class totals (km2) and per-class agreement |
| A3 | Which pixels changed, what they changed to, and why |
| A4 | Why agreement did not go up |
| A5 | Which codes are dropped in the one-sided pixels (incl. code 212) |
| B1 | Do area, edge, and pn now use the same valid pixels? |
| B2 | Python vs TerrSet, per metric |
| B3 | Threshold sweep, per metric |
| B4 | What the TS-only / TI-only pixels look like (z-scores, 1990-2020 change) |
| B5 | How much re-standardising edge moved the +/-0.5 cutoffs |

In [122]:
### Parameters ###
import os
os.environ["GDAL_CACHEMAX"] = "2048"   # MB; set before rasterio is imported. Lower if RAM is tight.

import csv, math, time
from pathlib import Path
from collections import Counter
import numpy as np, rasterio
from rasterio.windows import Window

D = Path("./data")
OLD_DIR = D / "5_Typology" / "TS_python_only_v2"     # previous TS version (before the pn mask)
F = {
    "old_ts_c":  OLD_DIR / "combined_timeseries" / "combined.tif",
    "old_ts_rc": OLD_DIR / "combined_rc_timeseries" / "combined_rc.tif",
    "new_ts_c":  D / "5_Typology" / "combined_timeseries" / "combined.tif",
    "new_ts_rc": D / "5_Typology" / "combined_rc_timeseries" / "combined_rc.tif",
    "ti_c":      D / "4_TS_and_TI" / "TI" / "1990-2020" / "combined.tif",
    "ti_rc":     D / "5_Typology" / "TimeInterval-1990-2020.tif",
    "mask":      D / "4_TS_and_TI" / "TS_mask" / "pn_validity_mask.tif",
}
METRICS = ("area", "edge", "pn")
TAU  = {m: D / "4_TS_and_TI" / "TS" / m / f"{m}_raster_group_MK_tau.tif" for m in METRICS}
Z    = {m: D / "4_TS_and_TI" / "TS_zscore" / m / f"{m}_MK_tau_z.tif" for m in METRICS}
TERR = {m: D / "4_TS_and_TI" / "terrset_version" / f"{m}_MK_tau_z.tif" for m in METRICS}
DIFF = {m: D / "4_TS_and_TI" / "TI" / "img_diff" / f"90-20_{m}.tif" for m in METRICS}
MW   = {m: sorted((D / "3_MovingWindow" / f"mw_{m}").glob(f"*_{m}_1km.tif")) for m in METRICS}
OUT  = D / "5_Typology" / "audit_pnmask"

THR = 0.5          # standardized-Tau threshold used by the remap
PX_KM2 = 0.0009    # km2 per 30 m pixel
ROWS = 128         # rows per processing strip; lower (e.g. 64) if RAM is tight

# Reference values from the saved outputs of the previous version
REF = {"pn_py_valid": 1_024_652_313, "pn_terr_only": 9_260_941, "pn_r": 1.00000, "pn_rmse": 0.00338}

In [123]:
### Typology tables and helpers ###
RECODE = {111:1,112:1,113:1, 121:2,122:2,123:2,131:2,132:2,133:2, 211:3, 213:4, 221:5,223:5,
          222:6,220:6, 231:7,232:7,233:7,230:7, 311:8,312:8,313:8, 321:9,322:9,323:9, 331:10,332:10,333:10}
NAMES = ["NoData", "Attrition", "Aggregation", "Shrinkage", "Perforation", "Deformation",
         "Persistent", "Enlargement", "Dissection", "Frag per se", "Creation"]
LUT = np.zeros(1001, np.int16)
for k, v in RECODE.items():
    LUT[k] = v
NC = 1001          # combined-code bins: bin = code + 1 (bin 0 = NoData)


def km2(n):
    return f"{n * PX_KM2:,.0f} km2"


def strips(src):
    for r in range(0, src.height, ROWS):
        yield Window(0, r, src.width, min(ROWS, src.height - r))


def read(src, w):
    """(data, valid) for one window."""
    a = src.read(1, window=w, masked=True)
    return np.ma.getdata(a), ~np.ma.getmaskarray(a)


def read_code(src, w):
    """Combined code as int32; NoData -> -1."""
    d, v = read(src, w)
    return np.where(v, d.astype(np.int32), -1)


def read_class(src, w):
    """Final typology class 1-10 as int16; NoData -> 0."""
    d, v = read(src, w)
    return np.where(v, d.astype(np.int16), 0)


def zero_or_missing(src, w):
    d, v = read(src, w)
    return (~v) | (d == 0)


def combined_class(code):
    return LUT[np.clip(code, 0, 1000)]


def has_zero_digit(c):
    return c > 0 and (c % 10 == 0 or (c // 10) % 10 == 0 or c < 100)


def lbl(c):
    if c < 0:
        return "NoData"
    if c == 0:
        return "0(background)"
    return f"{c}({NAMES[LUT[c]] if LUT[c] else 'dropped'})"


def write_csv(name, header, rows):
    OUT.mkdir(parents=True, exist_ok=True)
    with open(OUT / name, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)
    print(f"  saved {OUT / name}")

In [128]:
### 0) Files, grids, and run order ###
# img_diff files now live here; find each metric's file by name
### 0) Files, grids, and run order ###
DIFF_DIR = D / "4_TS_and_TI" / "TI" / "1990-2020" / "img_diff"
print("img_diff folder contents:", sorted(p.name for p in DIFF_DIR.glob("*.tif")))
for m in METRICS:
    hits = sorted(DIFF_DIR.glob(f"*{m}*.tif"))
    DIFF[m] = hits[0] if hits else DIFF_DIR / f"MISSING_{m}.tif"


paths = {**F,
         **{f"tau {m}": p for m, p in TAU.items()}, **{f"z {m}": p for m, p in Z.items()},
         **{f"terrset {m}": p for m, p in TERR.items()}, **{f"img_diff {m}": p for m, p in DIFF.items()},
         **{f"mw {m} first": (MW[m][0] if MW[m] else Path("missing")) for m in METRICS},
         **{f"mw {m} last": (MW[m][-1] if MW[m] else Path("missing")) for m in METRICS}}
grid = {}
for name, p in paths.items():
    if Path(p).exists():
        with rasterio.open(p) as s:
            grid[name] = (s.width, s.height)
        t = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"  OK      {name:16} {t}  {p}")
    else:
        print(f"  MISSING {name:16} {p}")
required = ["old_ts_c", "old_ts_rc", "new_ts_c", "new_ts_rc", "ti_c", "ti_rc", "mask"]
missing = [k for k in required if k not in grid]
assert not missing, f"required files missing: {missing}"
for m in METRICS:
    assert MW[m], f"no moving-window rasters found for {m}"
    print(f"  mw {m}: {len(MW[m])} years ({MW[m][0].name} ... {MW[m][-1].name})")
assert len(set(grid.values())) == 1, f"grids differ: {grid}"
print(f"  all {len(grid)} rasters are {next(iter(grid.values()))[0]} x {next(iter(grid.values()))[1]}")
HAVE = {k: k in grid for k in paths}
mt = lambda k: os.path.getmtime(paths[k])
for a, b in (("mask", "z edge"), ("z edge", "new_ts_c"), ("new_ts_c", "new_ts_rc")):
    if HAVE.get(a) and HAVE.get(b):
        ok = mt(b) >= mt(a)
        print(f"  {'OK   ' if ok else 'CHECK'} {b} is {'newer' if ok else 'OLDER'} than {a}")


img_diff folder contents: ['90-20_area.tif', '90-20_edge.tif', '90-20_pn.tif']
  OK      old_ts_c         2026-10-31 01:43  data/5_Typology/TS_python_only_v2/combined_timeseries/combined.tif
  OK      old_ts_rc        2026-10-31 01:45  data/5_Typology/TS_python_only_v2/combined_rc_timeseries/combined_rc.tif
  OK      new_ts_c         2026-11-02 00:10  data/5_Typology/combined_timeseries/combined.tif
  OK      new_ts_rc        2026-11-02 00:12  data/5_Typology/combined_rc_timeseries/combined_rc.tif
  OK      ti_c             2025-09-22 20:49  data/4_TS_and_TI/TI/1990-2020/combined.tif
  OK      ti_rc            2025-09-23 00:55  data/5_Typology/TimeInterval-1990-2020.tif
  OK      mask             2026-11-01 01:37  data/4_TS_and_TI/TS_mask/pn_validity_mask.tif
  OK      tau area         2026-11-01 22:09  data/4_TS_and_TI/TS/area/area_raster_group_MK_tau.tif
  OK      tau edge         2026-11-01 04:09  data/4_TS_and_TI/TS/edge/edge_raster_group_MK_tau.tif
  OK      tau pn           2026-

## Pass A: previous TS vs new TS vs TI
Reads the six typology rasters, the pn mask, the edge 1990/2020 rasters and the TI edge img_diff once.
Writes two maps to `audit_pnmask/`:

**`footprint_cause_new.tif`** (new TS vs TI): 1 = both classified, same class · 2 = both classified, different class ·
3 = TS-only, TI code 212 · 4 = TS-only, other · 5 = TI-only, TS code 212 · 6 = TI-only, other · 0 = neither

**`ts_change_old_vs_new.tif`** (previous TS vs new TS): 1 = newly classified, edge 0 at both endpoints (edge now kept) ·
2 = newly classified, other pixels · 3 = class changed, edge 0 at both endpoints · 4 = class changed, other pixels ·
5 = classified before, NoData now (expected: none) · 0 = no change

In [129]:
### Pass A ###
conf = {"old": np.zeros(121, np.int64), "new": np.zeros(121, np.int64)}   # TS class x TI class
trans = np.zeros(2 * 11 * 11 * 11, np.int64)        # cause x previous TS x new TS x TI
code_hist = {k: np.zeros(NC, np.int64) for k in ("old_ts", "new_ts", "ti")}
code_trans = np.zeros(NC * NC, np.int64)             # previous -> new TS combined code (changed only)
one_sided = {k: np.zeros(NC, np.int64) for k in
             ("previous TS-only: TI code", "new TS-only: TI code",
              "previous TI-only: TS code", "new TI-only: TS code")}
both212 = {"old": 0, "new": 0}
chk = Counter()          # all expected to be 0
ti_edge = Counter()      # how TI treats edge where edge = 0 at both endpoints
ti_edge_cls = np.zeros(11, np.int64)   # TI class at those pixels

src = {k: rasterio.open(F[k]) for k in ("old_ts_c", "old_ts_rc", "new_ts_c", "new_ts_rc", "ti_c", "ti_rc", "mask")}
src["e0"], src["e1"] = rasterio.open(MW["edge"][0]), rasterio.open(MW["edge"][-1])
src["diff_edge"] = rasterio.open(DIFF["edge"]) if HAVE["img_diff edge"] else None
OUT.mkdir(parents=True, exist_ok=True)
prof = src["new_ts_rc"].profile.copy()
prof.update(driver="GTiff", count=1, dtype="uint8", nodata=0, compress="deflate", tiled=False, blockysize=ROWS)
prof.pop("blockxsize", None)
fp_path, ch_path = OUT / "footprint_cause_new.tif", OUT / "ts_change_old_vs_new.tif"
t0 = time.time()
try:
    with rasterio.open(fp_path, "w", **prof) as fp_dst, rasterio.open(ch_path, "w", **prof) as ch_dst:
        windows = list(strips(src["new_ts_rc"]))
        for i, w in enumerate(windows, 1):
            oc, nc, tc = (read_code(src[k], w) for k in ("old_ts_c", "new_ts_c", "ti_c"))
            ok, nk, tk = (read_class(src[k], w) for k in ("old_ts_rc", "new_ts_rc", "ti_rc"))
            keep = read(src["mask"], w)[0] == 1
            E = zero_or_missing(src["e0"], w) & zero_or_missing(src["e1"], w)   # edge 0 at both endpoints
            ov, nv, tv = ok > 0, nk > 0, tk > 0

            chk["previous final != recode(previous combined)"] += int((combined_class(oc) != ok).sum())
            chk["new final != recode(new combined)"] += int((combined_class(nc) != nk).sum())
            chk["TI final != recode(TI combined)"] += int((combined_class(tc) != tk).sum())
            chk["new TS classified outside the pn mask"] += int((nv & ~keep).sum())
            chk["final classes outside 1-10"] += int(((ok > 10) | (nk > 10) | (tk > 10)).sum())

            for v, k in (("old", ok), ("new", nk)):
                conf[v] += np.bincount((k.astype(np.int64) * 11 + tk).ravel(), minlength=121)
            cause = np.where(E, 0, 1).astype(np.int64)
            trans += np.bincount((((cause * 11 + ok) * 11 + nk) * 11 + tk).ravel(), minlength=trans.size)
            for k, c in (("old_ts", oc), ("new_ts", nc), ("ti", tc)):
                code_hist[k] += np.bincount((np.clip(c, -1, NC - 2) + 1).ravel(), minlength=NC)
            d = oc != nc
            if d.any():
                code_trans += np.bincount((np.clip(oc[d], -1, NC - 2) + 1) * NC + np.clip(nc[d], -1, NC - 2) + 1,
                                          minlength=NC * NC)
            for name, sel, c in (("previous TS-only: TI code", ov & ~tv, tc), ("new TS-only: TI code", nv & ~tv, tc),
                                 ("previous TI-only: TS code", ~ov & tv, oc), ("new TI-only: TS code", ~nv & tv, nc)):
                if sel.any():
                    one_sided[name] += np.bincount(np.clip(c[sel], -1, NC - 2) + 1, minlength=NC)
            both212["old"] += int(((oc == 212) & (tc == 212)).sum())
            both212["new"] += int(((nc == 212) & (tc == 212)).sum())

            sel = E & keep
            ti_edge["edge 0 at both endpoints, inside the pn mask"] += int(sel.sum())
            ti_edge["... TI classified there"] += int((sel & tv).sum())
            ti_edge_cls += np.bincount(tk[sel], minlength=11)
            ti_edge["... TI combined edge digit = 0 (edge treated as NoData)"] += int((sel & (tc > 0) & (tc % 10 == 0)).sum())
            if src["diff_edge"] is not None:
                ti_edge["... TI edge img_diff = NoData"] += int((sel & ~read(src["diff_edge"], w)[1]).sum())

            fpm = np.zeros(nk.shape, np.uint8)
            fpm[nv & tv & (nk == tk)] = 1
            fpm[nv & tv & (nk != tk)] = 2
            fpm[nv & ~tv & (tc == 212)] = 3
            fpm[nv & ~tv & (tc != 212)] = 4
            fpm[~nv & tv & (nc == 212)] = 5
            fpm[~nv & tv & (nc != 212)] = 6
            fp_dst.write(fpm, 1, window=w)
            chm = np.zeros(nk.shape, np.uint8)
            chm[~ov & nv & E] = 1
            chm[~ov & nv & ~E] = 2
            chm[ov & nv & (ok != nk) & E] = 3
            chm[ov & nv & (ok != nk) & ~E] = 4
            chm[ov & ~nv] = 5
            ch_dst.write(chm, 1, window=w)
            if i % max(1, len(windows) // 20) == 0 or i == len(windows):
                print(f"  Pass A: {i:,}/{len(windows):,} strips ({(time.time() - t0) / 60:.1f} min)")
finally:
    for s in src.values():
        if s is not None:
            s.close()
trans = trans.reshape(2, 11, 11, 11)
code_trans = code_trans.reshape(NC, NC)
co, cn = conf["old"].reshape(11, 11), conf["new"].reshape(11, 11)
print(f"maps written:\n  {fp_path}\n  {ch_path}")

  Pass A: 37/755 strips (0.9 min)
  Pass A: 74/755 strips (1.7 min)
  Pass A: 111/755 strips (2.5 min)
  Pass A: 148/755 strips (3.3 min)
  Pass A: 185/755 strips (4.2 min)
  Pass A: 222/755 strips (5.0 min)
  Pass A: 259/755 strips (5.9 min)
  Pass A: 296/755 strips (6.7 min)
  Pass A: 333/755 strips (7.5 min)
  Pass A: 370/755 strips (8.4 min)
  Pass A: 407/755 strips (9.3 min)
  Pass A: 444/755 strips (10.2 min)
  Pass A: 481/755 strips (11.3 min)
  Pass A: 518/755 strips (12.3 min)
  Pass A: 555/755 strips (13.3 min)
  Pass A: 592/755 strips (14.2 min)
  Pass A: 629/755 strips (15.2 min)
  Pass A: 666/755 strips (16.1 min)
  Pass A: 703/755 strips (16.9 min)
  Pass A: 740/755 strips (17.8 min)
  Pass A: 755/755 strips (18.1 min)
maps written:
  data/5_Typology/audit_pnmask/footprint_cause_new.tif
  data/5_Typology/audit_pnmask/ts_change_old_vs_new.tif


### A1) Footprint and agreement
**Look for:** TI valid identical (TI was not rerun) · every consistency check 0 · the agreement change shown to
4 decimals, so the actual difference is visible rather than rounded to 56.8%.

In [130]:
### A1) Footprint and agreement: previous vs new ###
def footprint(m):
    both, ts_only, ti_only = int(m[1:, 1:].sum()), int(m[1:, 0].sum()), int(m[0, 1:].sum())
    return dict(ts=both + ts_only, ti=both + ti_only, both=both, ts_only=ts_only, ti_only=ti_only,
                nonoverlap=ts_only + ti_only, agree=int(np.trace(m[1:, 1:])))

fo, fn = footprint(co), footprint(cn)
print(f"{'':18}{'previous':>16}{'new':>16}{'change':>14}")
for label, k in (("TS valid", "ts"), ("TI valid", "ti"), ("both valid", "both"), ("TS-only", "ts_only"),
                 ("TI-only", "ti_only"), ("non-overlap", "nonoverlap"), ("agreeing pixels", "agree")):
    print(f"{label:18}{fo[k]:>16,}{fn[k]:>16,}{fn[k] - fo[k]:>+14,}   ({km2(fn[k] - fo[k])})")
ao, an = fo["agree"] / fo["both"], fn["agree"] / fn["both"]
print(f"{'agreement':18}{ao:>16.4%}{an:>16.4%}{(an - ao) * 100:>+13.4f} percentage points")
print("\nTI valid identical:", "OK" if fo["ti"] == fn["ti"] else "CHANGED - check the TI file")
print("\nConsistency checks (all should be 0):")
for k, v in chk.items():
    print(f"  {'OK   ' if v == 0 else 'CHECK'} {k:48} {v:,}")

                          previous             new        change
TS valid             1,000,130,520   1,000,246,561      +116,041   (104 km2)
TI valid             1,019,081,899   1,019,081,899            +0   (0 km2)
both valid             996,647,176     996,757,493      +110,317   (99 km2)
TS-only                  3,483,344       3,489,068        +5,724   (5 km2)
TI-only                 22,434,723      22,324,406      -110,317   (-99 km2)
non-overlap             25,918,067      25,813,474      -104,593   (-94 km2)
agreeing pixels        566,475,640     566,344,299      -131,341   (-118 km2)
agreement                 56.8381%        56.8187%      -0.0195 percentage points

TI valid identical: OK

Consistency checks (all should be 0):
  OK    previous final != recode(previous combined)      0
  OK    new final != recode(new combined)                0
  OK    TI final != recode(TI combined)                  0
  OK    new TS classified outside the pn mask            0
  OK    final class

### A2) Class totals and per-class agreement
*TS-side agreement*: of the pixels TS puts in a class (where TI is classified), the share TI puts in the same class.
*TI-side agreement*: the same from TI's side.

In [131]:
### A2) Class totals (km2) and per-class agreement ###
print(f"{'class':13}{'TS prev':>10}{'TS new':>10}{'change':>9}{'TI':>10}   "
      f"{'TS-side agreement':>22}   {'TI-side agreement':>22}")
rows = []
for k in range(1, 11):
    ts_o, ts_n, ti = int(co[k, :].sum()), int(cn[k, :].sum()), int(cn[:, k].sum())
    us = [m[k, k] / max(m[k, 1:].sum(), 1) for m in (co, cn)]
    pr = [m[k, k] / max(m[1:, k].sum(), 1) for m in (co, cn)]
    print(f"{NAMES[k]:13}{ts_o * PX_KM2:>10,.0f}{ts_n * PX_KM2:>10,.0f}{(ts_n - ts_o) * PX_KM2:>+9,.0f}"
          f"{ti * PX_KM2:>10,.0f}   {us[0]:>9.1%} -> {us[1]:>7.1%}      {pr[0]:>9.1%} -> {pr[1]:>7.1%}")
    rows.append([NAMES[k], ts_o, ts_n, ti, int(co[k, k]), int(cn[k, k])])
print("(km2)")
write_csv("class_totals_pixels.csv",
          ["class", "TS_previous_px", "TS_new_px", "TI_px", "agree_previous_px", "agree_new_px"], rows)
for v, m in (("previous", co), ("new", cn)):
    write_csv(f"confusion_TS_{v}_vs_TI_pixels.csv", ["TS_class \\ TI_class"] + NAMES,
              [[NAMES[i]] + [int(x) for x in m[i]] for i in range(11)])

class           TS prev    TS new   change        TI        TS-side agreement        TI-side agreement
Attrition       124,046   124,046       +1   167,558       78.5% ->   78.5%          59.1% ->   59.1%
Aggregation     159,537   159,537       +0   125,986       58.0% ->   58.0%          73.6% ->   73.6%
Shrinkage        63,443    63,443       +0    66,209       36.9% ->   36.9%          37.3% ->   37.3%
Perforation      16,670    16,774     +104    27,327       54.5% ->   54.4%          38.9% ->   39.0%
Deformation      32,186    32,559     +374     2,063        1.9% ->    1.9%          30.4% ->   30.6%
Persistent       97,571    97,198     -374    23,319       22.0% ->   21.9%          91.7% ->   90.9%
Enlargement      97,290    97,290       +0   101,108       44.6% ->   44.6%          43.2% ->   43.2%
Dissection       97,343    97,343       +0   169,205       88.3% ->   88.3%          52.9% ->   52.9%
Frag per se      59,176    59,176       +0     1,222        0.8% ->    0.8%      

### A3) Every pixel whose TS result changed
Split by cause: pixels where **edge is 0 at both 1990 and 2020** (edge was dropped before and is now kept by the pn
mask) vs **all other pixels** (only the edge z-score standardisation changed there). Each line shows how many of
those pixels matched TI before and after.

In [132]:
### A3) What changed, and why ###
CAUSE = ["edge 0 at both endpoints (edge now kept)", "other pixels (edge z-score shift)"]
for c in range(2):
    on = trans[c].sum(axis=2)            # previous x new
    newly, dropped = int(on[0, 1:].sum()), int(on[1:, 0].sum())
    changed = int(on[1:, 1:].sum() - np.trace(on[1:, 1:]))
    print(f"\n{CAUSE[c]}: newly classified {newly:,} | NoData now {dropped:,} | class changed {changed:,}")
    pairs = sorted(((int(on[o, n]), o, n) for o in range(11) for n in range(11) if o != n and on[o, n]), reverse=True)
    for cnt, o, n in pairs[:12]:
        t = trans[c, o, n]
        print(f"   {NAMES[o]:12} -> {NAMES[n]:12} {cnt:>11,} ({km2(cnt):>9})   matched TI: "
              f"{int(t[o]) if o else 0:,} before -> {int(t[n]) if n else 0:,} now   "
              f"| TI says: " + ", ".join(f"{NAMES[x]} {int(t[x]):,}" for x in np.argsort(t)[::-1][:3] if t[x]))
write_csv("ts_class_changes_by_cause.csv", ["cause", "TS_previous", "TS_new", "TI", "pixels"],
          [[CAUSE[c], NAMES[o], NAMES[n], NAMES[x], int(trans[c, o, n, x])]
           for c in range(2) for o in range(11) for n in range(11) for x in range(11)
           if o != n and trans[c, o, n, x]])

print("\nCombined-code changes, previous TS -> new TS (top 25):")
top = sorted(((int(code_trans[a, b]), a - 1, b - 1) for a, b in np.argwhere(code_trans > 0)), reverse=True)
for n, a, b in top[:25]:
    print(f"   {lbl(a):18} -> {lbl(b):18} {n:>11,}")
write_csv("ts_combined_code_changes.csv", ["previous_code", "new_code", "pixels"], [[a, b, n] for n, a, b in top])


edge 0 at both endpoints (edge now kept): newly classified 24,049 | NoData now 0 | class changed 156,278
   Persistent   -> Deformation      156,278 (  141 km2)   matched TI: 156,264 before -> 0 now   | TI says: Persistent 156,264, Enlargement 7, NoData 7
   NoData       -> Perforation       23,307 (   21 km2)   matched TI: 0 before -> 0 now   | TI says: Persistent 23,305, NoData 2
   NoData       -> Attrition            604 (    1 km2)   matched TI: 0 before -> 0 now   | TI says: NoData 604
   NoData       -> Aggregation           58 (    0 km2)   matched TI: 0 before -> 0 now   | TI says: Persistent 49, NoData 9
   NoData       -> Creation              53 (    0 km2)   matched TI: 0 before -> 0 now   | TI says: NoData 53
   NoData       -> Shrinkage             14 (    0 km2)   matched TI: 0 before -> 0 now   | TI says: NoData 12, Persistent 2
   NoData       -> Frag per se           13 (    0 km2)   matched TI: 0 before -> 0 now   | TI says: NoData 13

other pixels (edge z-score sh

### A4) Why agreement did not go up
Agreement = pixels where TS and TI give the same class ÷ pixels both classify. This splits the net change in
agreeing pixels by cause, and shows how TI treats edge where edge is 0 at both endpoints.

In [133]:
### A4) Agreement change, by cause ###
o_, n_, t_ = np.meshgrid(np.arange(11), np.arange(11), np.arange(11), indexing="ij")
agree_o, agree_n = (o_ > 0) & (o_ == t_), (n_ > 0) & (n_ == t_)
groups = {"newly classified": (o_ == 0) & (n_ > 0),
          "class changed": (o_ > 0) & (n_ > 0) & (o_ != n_),
          "NoData now": (o_ > 0) & (n_ == 0)}
print(f"{'':58}{'pixels':>12}{'agreed before':>15}{'agree now':>12}{'net':>11}")
net = 0
for c in range(2):
    for g, sel in groups.items():
        t = trans[c]
        px, a0, a1 = int(t[sel].sum()), int(t[sel & agree_o].sum()), int(t[sel & agree_n].sum())
        net += a1 - a0
        if px:
            print(f"{CAUSE[c] + ': ' + g:58}{px:>12,}{a0:>15,}{a1:>12,}{a1 - a0:>+11,}")
print(f"{'net change in agreeing pixels':58}{'':>39}{net:>+11,}")
assert net == fn["agree"] - fo["agree"], "breakdown does not add up"
print(f"\nagreement before: {fo['agree']:,} / {fo['both']:,} = {ao:.4%}")
print(f"agreement now   : {fn['agree']:,} / {fn['both']:,} = {an:.4%}")

print("\nHow TI treats edge where edge = 0 at both 1990 and 2020 (inside the pn mask):")
base = max(ti_edge["edge 0 at both endpoints, inside the pn mask"], 1)
for k, v in ti_edge.items():
    print(f"  {k:58} {v:>14,}  ({v / base:.1%})")
t = ti_edge_cls
print("  TI class at these pixels: " + ", ".join(f"{NAMES[x]} {int(t[x]):,}" for x in np.argsort(t)[::-1] if t[x]))

                                                                pixels  agreed before   agree now        net
edge 0 at both endpoints (edge now kept): newly classified      24,049              0           0         +0
edge 0 at both endpoints (edge now kept): class changed        156,278        156,264           0   -156,264
other pixels (edge z-score shift): newly classified             91,992              0      39,725    +39,725
other pixels (edge z-score shift): class changed               258,971         17,904       3,102    -14,802
net change in agreeing pixels                                                                       -131,341

agreement before: 566,475,640 / 996,647,176 = 56.8381%
agreement now   : 566,344,299 / 996,757,493 = 56.8187%

How TI treats edge where edge = 0 at both 1990 and 2020 (inside the pn mask):
  edge 0 at both endpoints, inside the pn mask                    7,372,019  (100.0%)
  ... TI classified there                                         7,37

### A5) One-sided pixels and dropped codes
Equivalent of the earlier cells 7, 8, 9, 15 and 20, for the previous and the new TS.

In [134]:
### A5) Which codes are dropped? ###
for name, h in one_sided.items():
    tot = int(h.sum())
    print(f"\n{name}: {tot:,} px ({km2(tot)})")
    for b in np.argsort(h)[::-1][:8]:
        if h[b] == 0:
            break
        print(f"   {lbl(b - 1):18} {int(h[b]):>12,} ({km2(int(h[b]))})")

print("\nCode 212:")
for v, key in (("previous", "old_ts"), ("new", "new_ts")):
    vv = "old" if v == "previous" else "new"
    print(f"  {v:8} TS 212 {int(code_hist[key][213]):>12,} ({km2(int(code_hist[key][213]))}) | "
          f"TI 212 {int(code_hist['ti'][213]):,} | 212 in both {both212[vv]:,} ({km2(both212[vv])})")

print("\nCombined codes dropped by the recode map, whole map (background 0 excluded):")
for key, label in (("old_ts", "previous TS"), ("new_ts", "new TS"), ("ti", "TI")):
    h = code_hist[key]
    dropped = sorted(((int(h[c + 1]), c) for c in range(1, 1000) if h[c + 1] and LUT[c] == 0), reverse=True)
    zero = sum(n for n, c in dropped if has_zero_digit(c))
    print(f"  {label:12} {sum(n for n, _ in dropped):>12,} px | with a 0 digit {zero:>9,} | "
          + ", ".join(f"{c}: {n:,}" for n, c in dropped[:6]))

for label, row in (("TI class of new TI-only pixels", cn[0, :]), ("TS class of new TS-only pixels", cn[:, 0])):
    print(f"\n{label}: " + ", ".join(f"{NAMES[k]} {int(row[k]):,}" for k in np.argsort(row)[::-1] if k and row[k]))


previous TS-only: TI code: 3,483,344 px (3,135 km2)
   212(dropped)          3,483,294 (3,135 km2)
   330(dropped)                 25 (0 km2)
   210(dropped)                 15 (0 km2)
   110(dropped)                 10 (0 km2)

new TS-only: TI code: 3,489,068 px (3,140 km2)
   212(dropped)          3,488,325 (3,139 km2)
   110(dropped)                632 (1 km2)
   330(dropped)                 93 (0 km2)
   210(dropped)                 18 (0 km2)

previous TI-only: TS code: 22,434,723 px (20,191 km2)
   212(dropped)         22,378,674 (20,141 km2)
   210(dropped)             56,000 (50 km2)
   120(dropped)                 49 (0 km2)

new TI-only: TS code: 22,324,406 px (20,092 km2)
   212(dropped)         22,324,406 (20,092 km2)

Code 212:
  previous TS 212   24,465,030 (22,019 km2) | TI 212 5,569,650 | 212 in both 2,086,356 (1,878 km2)
  new      TS 212   24,405,752 (21,965 km2) | TI 212 5,569,650 | 212 in both 2,081,325 (1,873 km2)

Combined codes dropped by the recode map, whole m

## Pass B: per-metric checks
For each of area, edge, and pn, reads the Tau, z-score, TerrSet, TI img_diff, and 1990/2020 moving-window rasters once.
Groups (TS-only, TI-only, both) use the **new** TS final typology vs TI.

In [135]:
### Pass B ###
SWEEP = (0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5)
GROUPS = ("both valid", "TS-only", "TI-only")
B = {}
for m in METRICS:
    if not (HAVE[f"tau {m}"] and HAVE[f"z {m}"]):
        print(f"{m}: Tau or z-score missing, skipped")
        continue
    r = B[m] = dict(tau_total=0, tau_valid=0, tau_zero=0, tau_outside_mask=0, z_valid=0, z_vs_tau=0,
                    z_sum=0.0, z_sq=0.0, terr=Counter() if HAVE[f"terrset {m}"] else None,
                    sweep=np.zeros((len(SWEEP), 2), np.int64), groups={g: Counter() for g in GROUPS},
                    edge=dict(n_all=0, s_all=0.0, q_all=0.0, n_E=0, s_E=0.0, q_E=0.0, band=Counter()))
    s = {"tau": rasterio.open(TAU[m]), "z": rasterio.open(Z[m]), "mask": rasterio.open(F["mask"]),
         "ts": rasterio.open(F["new_ts_rc"]), "ti": rasterio.open(F["ti_rc"]),
         "m0": rasterio.open(MW[m][0]), "m1": rasterio.open(MW[m][-1])}
    if r["terr"] is not None:
        s["terr"] = rasterio.open(TERR[m])
    if HAVE[f"img_diff {m}"]:
        s["diff"] = rasterio.open(DIFF[m])
    t0 = time.time()
    try:
        windows = list(strips(s["tau"]))
        for i, w in enumerate(windows, 1):
            tau, tv = read(s["tau"], w); tau = tau.astype(np.float64)
            z, zv = read(s["z"], w); z = z.astype(np.float64)
            keep = read(s["mask"], w)[0] == 1
            ts_ok, ti_ok = read(s["ts"], w)[1], read(s["ti"], w)[1]
            a0, v0 = read(s["m0"], w); a1, v1 = read(s["m1"], w)
            a0, a1 = a0.astype(np.float64), a1.astype(np.float64)

            r["tau_total"] += tau.size
            r["tau_valid"] += int(tv.sum())
            r["tau_zero"] += int((tv & (tau == 0)).sum())
            r["tau_outside_mask"] += int((tv & ~keep).sum())
            r["z_valid"] += int(zv.sum())
            r["z_vs_tau"] += int((zv != tv).sum())
            r["z_sum"] += float(z[zv].sum()); r["z_sq"] += float((z[zv] ** 2).sum())

            if "terr" in s:
                y, yv = read(s["terr"], w); y = y.astype(np.float64)
                T, b = r["terr"], zv & yv
                T["py_only"] += int((zv & ~yv).sum()); T["terr_only"] += int((~zv & yv).sum()); T["both"] += int(b.sum())
                T["ty_n"] += int(yv.sum()); T["ty_s"] += float(y[yv].sum()); T["ty_q"] += float((y[yv] ** 2).sum())
                if b.any():
                    x_, y_ = z[b], y[b]
                    dd = x_ - y_
                    T["sd"] += float(dd.sum()); T["sd2"] += float((dd * dd).sum())
                    T["max"] = max(T["max"], float(np.abs(dd).max()))
                    T["sx"] += float(x_.sum()); T["sy"] += float(y_.sum())
                    T["sxx"] += float((x_ * x_).sum()); T["syy"] += float((y_ * y_).sum()); T["sxy"] += float((x_ * y_).sum())

            if "diff" in s:
                dv, dvv = read(s["diff"], w)
                sel = zv & dvv
                if sel.any():
                    ti_dir, zs = np.sign(dv[sel]), z[sel]
                    for k, thr in enumerate(SWEEP):
                        ts_dir = np.where(zs > thr, 1, np.where(zs < -thr, -1, 0))
                        r["sweep"][k, 0] += zs.size
                        r["sweep"][k, 1] += int((ts_dir == ti_dir).sum())

            ends = v0 & v1
            for g, gm in zip(GROUPS, (ts_ok & ti_ok, ts_ok & ~ti_ok, ~ts_ok & ti_ok)):
                G = r["groups"][g]
                G["px"] += int(gm.sum())
                G["z_nodata"] += int((gm & ~zv).sum())
                gz = gm & zv
                if gz.any():
                    zz = z[gz]
                    G["z_n"] += zz.size; G["z_sum"] += float(zz.sum())
                    G["z_stable"] += int((np.abs(zz) <= THR).sum())
                    G["z_inc"] += int((zz > THR).sum()); G["z_dec"] += int((zz < -THR).sum())
                ge = gm & ends
                if ge.any():
                    net, st, en = (a1 - a0)[ge], a0[ge], a1[ge]
                    G["e_n"] += net.size; G["net_sum"] += float(net.sum()); G["abs_sum"] += float(np.abs(net).sum())
                    G["nochange"] += int((net == 0).sum())
                    G["small"] += int(((st > 0) & (np.abs(net) < 0.1 * st)).sum())
                    G["zero_both"] += int(((st == 0) & (en == 0)).sum())

            if m == "edge":
                E = (~v0 | (a0 == 0)) & (~v1 | (a1 == 0))
                Ed = r["edge"]
                x, xe = tau[tv], tau[tv & E]
                Ed["n_all"] += x.size; Ed["s_all"] += float(x.sum()); Ed["q_all"] += float((x * x).sum())
                Ed["n_E"] += xe.size; Ed["s_E"] += float(xe.sum()); Ed["q_E"] += float((xe * xe).sum())
                band = tv & ~E & (((tau > 0.2) & (tau < 0.4)) | ((tau > -0.3) & (tau < -0.1)))
                if band.any():
                    vals, cnt = np.unique(tau[band].astype(np.float32), return_counts=True)
                    for v_, c_ in zip(vals, cnt):
                        Ed["band"][float(v_)] += int(c_)
            if i % max(1, len(windows) // 10) == 0 or i == len(windows):
                print(f"  Pass B {m}: {i:,}/{len(windows):,} strips ({(time.time() - t0) / 60:.1f} min)")
    finally:
        for x_ in s.values():
            x_.close()

  Pass B area: 75/755 strips (1.1 min)
  Pass B area: 150/755 strips (2.3 min)
  Pass B area: 225/755 strips (3.5 min)
  Pass B area: 300/755 strips (4.8 min)
  Pass B area: 375/755 strips (6.1 min)
  Pass B area: 450/755 strips (7.7 min)
  Pass B area: 525/755 strips (9.6 min)
  Pass B area: 600/755 strips (11.2 min)
  Pass B area: 675/755 strips (12.7 min)
  Pass B area: 750/755 strips (13.9 min)
  Pass B area: 755/755 strips (14.0 min)
  Pass B edge: 75/755 strips (1.0 min)
  Pass B edge: 150/755 strips (2.3 min)
  Pass B edge: 225/755 strips (3.5 min)
  Pass B edge: 300/755 strips (4.8 min)
  Pass B edge: 375/755 strips (6.3 min)
  Pass B edge: 450/755 strips (7.9 min)
  Pass B edge: 525/755 strips (10.0 min)
  Pass B edge: 600/755 strips (11.7 min)
  Pass B edge: 675/755 strips (13.2 min)
  Pass B edge: 750/755 strips (14.5 min)
  Pass B edge: 755/755 strips (14.5 min)
  Pass B pn: 75/755 strips (0.8 min)
  Pass B pn: 150/755 strips (1.7 min)
  Pass B pn: 225/755 strips (2.8 min)


### B1) Tau and z-score footprint per metric
**Look for:** the same Tau-valid count for all three metrics, 0 outside the mask, pn unchanged from the previous run.

In [136]:
### B1) Tau / z-score footprint ###
print(f"{'metric':7}{'Tau valid':>16}{'Tau exactly 0':>15}{'NoData':>8}{'outside mask':>14}"
      f"{'z valid':>16}{'z vs Tau':>10}{'z mean':>9}{'z std':>8}")
for m, r in B.items():
    n = max(r["z_valid"], 1); zm = r["z_sum"] / n; zs = math.sqrt(max(r["z_sq"] / n - zm * zm, 0))
    print(f"{m:7}{r['tau_valid']:>16,}{r['tau_zero']:>15,}{1 - r['tau_valid'] / r['tau_total']:>8.1%}"
          f"{r['tau_outside_mask']:>14,}{r['z_valid']:>16,}{r['z_vs_tau']:>10,}{zm:>+9.4f}{zs:>8.4f}")
same = len({r["tau_valid"] for r in B.values()}) == 1
print("\n" + ("OK: all metrics use the same valid pixels" if same else "CHECK: valid pixel counts differ between metrics"))
if "pn" in B:
    print(f"pn Tau valid {B['pn']['tau_valid']:,} vs previous {REF['pn_py_valid']:,}: "
          f"{'unchanged' if B['pn']['tau_valid'] == REF['pn_py_valid'] else 'CHANGED'}")

metric        Tau valid  Tau exactly 0  NoData  outside mask         z valid  z vs Tau   z mean   z std
area      1,024,652,313     19,060,926   85.8%             0   1,024,652,313         0  +0.0000  1.0000
edge      1,024,652,313     22,187,080   85.8%             0   1,024,652,313         0  +0.0000  1.0000
pn        1,024,652,313     94,760,790   85.8%             0   1,024,652,313         0  +0.0000  1.0000

OK: all metrics use the same valid pixels
pn Tau valid 1,024,652,313 vs previous 1,024,652,313: unchanged


In [137]:
### B2) Python standardized Tau vs TerrSet ###
for m, r in B.items():
    T = r["terr"]
    if T is None or not T["both"]:
        print(f"{m}: no TerrSet file or no overlap, skipped")
        continue
    n = T["both"]
    den = math.sqrt((n * T["sxx"] - T["sx"] ** 2) * (n * T["syy"] - T["sy"] ** 2))
    rr = (n * T["sxy"] - T["sx"] * T["sy"]) / den if den else float("nan")
    tm = T["ty_s"] / T["ty_n"]; tsd = math.sqrt(max(T["ty_q"] / T["ty_n"] - tm * tm, 0))
    print(f"{m}: Python {r['z_valid']:,} | TerrSet {T['ty_n']:,} | both {n:,} | "
          f"Python-only {T['py_only']:,} | TerrSet-only {T['terr_only']:,}")
    print(f"      TerrSet mean {tm:+.4f} std {tsd:.4f} | mean diff {T['sd'] / n:+.5f} | "
          f"RMSE {math.sqrt(T['sd2'] / n):.5f} | max |diff| {T['max']:.5f} | r {rr:.5f}")
print(f"\npn previous: Python {REF['pn_py_valid']:,} | TerrSet-only {REF['pn_terr_only']:,} | "
      f"r {REF['pn_r']:.5f} | RMSE {REF['pn_rmse']}")
print("edge now keeps full-forest-window pixels (edge = 0 at both endpoints), so its footprint vs TerrSet is expected to change.")

area: Python 1,024,652,313 | TerrSet 1,033,913,783 | both 1,024,652,313 | Python-only 0 | TerrSet-only 9,261,470
      TerrSet mean +0.0000 std 1.0000 | mean diff -0.00072 | RMSE 0.00391 | max |diff| 0.00758 | r 1.00000
edge: Python 1,024,652,313 | TerrSet 1,027,646,953 | both 1,018,385,512 | Python-only 6,266,801 | TerrSet-only 9,261,441
      TerrSet mean +0.0000 std 1.0000 | mean diff +0.00017 | RMSE 0.07932 | max |diff| 0.22698 | r 0.99687
pn: Python 1,024,652,313 | TerrSet 1,033,913,254 | both 1,024,652,313 | Python-only 0 | TerrSet-only 9,260,941
      TerrSet mean -0.0000 std 1.0000 | mean diff -0.00119 | RMSE 0.00338 | max |diff| 0.00888 | r 1.00000

pn previous: Python 1,024,652,313 | TerrSet-only 9,260,941 | r 1.00000 | RMSE 0.00338
edge now keeps full-forest-window pixels (edge = 0 at both endpoints), so its footprint vs TerrSet is expected to change.


In [138]:
### B3) Threshold sweep: TS direction vs TI img_diff sign ###
print(f"{'threshold':>10}" + "".join(f"{m:>10}" for m in B))
for k, thr in enumerate(SWEEP):
    print(f"{'+/-' + str(thr):>10}" + "".join(
        f"{(r['sweep'][k, 1] / r['sweep'][k, 0] if r['sweep'][k, 0] else float('nan')):>10.1%}" for r in B.values()))

 threshold      area      edge        pn
    +/-0.0     86.0%     84.0%     66.8%
    +/-0.1     85.4%     81.7%     66.6%
   +/-0.25     81.1%     79.0%     74.9%
    +/-0.5     71.9%     69.8%     71.6%
   +/-0.75     59.8%     58.1%     66.5%
    +/-1.0     45.3%     44.4%     58.9%
    +/-1.5     14.6%     16.1%     38.7%


In [139]:
### B4) What the TS-only / TI-only / both-valid pixels look like ###
for g in GROUPS:
    print(f"\n=== {g} (new TS vs TI) ===")
    for m, r in B.items():
        G = r["groups"][g]
        if not G["px"]:
            continue
        zn, en = max(G["z_n"], 1), max(G["e_n"], 1)
        print(f"  {m:4} {G['px']:,} px ({km2(G['px'])}) | z mean {G['z_sum'] / zn:+.3f}: stable {G['z_stable'] / zn:.1%}, "
              f"increase {G['z_inc'] / zn:.1%}, decrease {G['z_dec'] / zn:.1%}, z NoData {G['z_nodata']:,}")
        print(f"       1990->2020: mean net {G['net_sum'] / en:+.2f}, mean |net| {G['abs_sum'] / en:.2f}, "
              f"no change {G['nochange'] / en:.1%}, |net| < 10% of 1990 {G['small'] / en:.1%}, 0 at both ends {G['zero_both']:,}")


=== both valid (new TS vs TI) ===
  area 996,757,493 px (897,082 km2) | z mean +0.024: stable 28.1%, increase 38.3%, decrease 33.5%, z NoData 0
       1990->2020: mean net -30.32, mean |net| 164.01, no change 3.0%, |net| < 10% of 1990 28.0%, 0 at both ends 0
  edge 996,757,493 px (897,082 km2) | z mean +0.002: stable 27.2%, increase 37.5%, decrease 35.3%, z NoData 0
       1990->2020: mean net +1.66, mean |net| 48.62, no change 3.6%, |net| < 10% of 1990 28.0%, 0 at both ends 7,338,562
  pn   996,757,493 px (897,082 km2) | z mean +0.001: stable 34.0%, increase 34.5%, decrease 31.6%, z NoData 0
       1990->2020: mean net +0.42, mean |net| 1.99, no change 23.5%, |net| < 10% of 1990 25.4%, 0 at both ends 0

=== TS-only (new TS vs TI) ===
  area 3,489,068 px (3,140 km2) | z mean -0.322: stable 65.8%, increase 4.8%, decrease 29.4%, z NoData 0
       1990->2020: mean net -13.50, mean |net| 13.50, no change 0.0%, |net| < 10% of 1990 86.7%, 0 at both ends 0
  edge 3,489,068 px (3,140 km2) | z

In [140]:
### B5) Edge: how far re-standardising moved the +/-0.5 cutoffs ###
if "edge" in B:
    Ed = B["edge"]["edge"]
    n_new, n_old = Ed["n_all"], Ed["n_all"] - Ed["n_E"]
    m_new = Ed["s_all"] / n_new; sd_new = math.sqrt(Ed["q_all"] / n_new - m_new ** 2)
    m_old = (Ed["s_all"] - Ed["s_E"]) / n_old; sd_old = math.sqrt((Ed["q_all"] - Ed["q_E"]) / n_old - m_old ** 2)
    up_o, up_n, lo_o, lo_n = m_old + THR * sd_old, m_new + THR * sd_new, m_old - THR * sd_old, m_new - THR * sd_new
    print(f"edge Tau previous: n={n_old:,} mean={m_old:.6f} std={sd_old:.6f} (rebuilt)")
    print(f"edge Tau new     : n={n_new:,} mean={m_new:.6f} std={sd_new:.6f}")
    print(f"edge kept by the pn mask (0 at both endpoints): {Ed['n_E']:,} ({km2(Ed['n_E'])})")
    print(f"'increase' cutoff in raw Tau: {up_o:.5f} -> {up_n:.5f}")
    print(f"'decrease' cutoff in raw Tau: {lo_o:.5f} -> {lo_n:.5f}")
    fu = {v: c for v, c in Ed["band"].items() if (v > up_o) != (v > up_n)}
    fl = {v: c for v, c in Ed["band"].items() if (v < lo_o) != (v < lo_n)}
    print(f"{'stable -> increase' if up_n < up_o else 'increase -> stable'}: {sum(fu.values()):,} px "
          f"at Tau {', '.join(f'{v:.5f} (= {v * 465:.1f}/465)' for v in sorted(fu))}")
    print(f"{'stable -> decrease' if lo_n > lo_o else 'decrease -> stable'}: {sum(fl.values()):,} px "
          + (f"at Tau {', '.join(f'{v:.5f}' for v in sorted(fl))}" if fl else ""))

edge Tau previous: n=1,017,280,294 mean=0.057552 std=0.479317 (rebuilt)
edge Tau new     : n=1,024,652,313 mean=0.057089 std=0.477651
edge kept by the pn mask (0 at both endpoints): 7,372,019 (6,635 km2)
'increase' cutoff in raw Tau: 0.29721 -> 0.29591
'decrease' cutoff in raw Tau: -0.18211 -> -0.18174
stable -> increase: 1,423,799 px at Tau 0.29677 (= 138.0/465)
stable -> decrease: 0 px 


In [141]:
### Summary: previous vs new ###
h_o, h_n = code_hist["old_ts"], code_hist["new_ts"]
zero_o = sum(int(h_o[c + 1]) for c in range(1, 1000) if LUT[c] == 0 and has_zero_digit(c))
zero_n = sum(int(h_n[c + 1]) for c in range(1, 1000) if LUT[c] == 0 and has_zero_digit(c))
rows = [("TS valid px", fo["ts"], fn["ts"]), ("TI valid px", fo["ti"], fn["ti"]),
        ("both valid px", fo["both"], fn["both"]), ("TS-only px", fo["ts_only"], fn["ts_only"]),
        ("TI-only px", fo["ti_only"], fn["ti_only"]), ("non-overlap px", fo["nonoverlap"], fn["nonoverlap"]),
        ("agreeing px", fo["agree"], fn["agree"]),
        ("TS code 212 px", int(h_o[213]), int(h_n[213])),
        ("TS codes with a 0 digit px", zero_o, zero_n),
        ("TS Persistent px", int(co[6, :].sum()), int(cn[6, :].sum())),
        ("TS Deformation px", int(co[5, :].sum()), int(cn[5, :].sum())),
        ("TS Perforation px", int(co[4, :].sum()), int(cn[4, :].sum()))]
print(f"{'':28}{'previous':>16}{'new':>16}{'change':>14}")
for label, a, b in rows:
    print(f"{label:28}{a:>16,}{b:>16,}{b - a:>+14,}")
print(f"{'agreement':28}{ao:>16.4%}{an:>16.4%}{(an - ao) * 100:>+13.4f} pts")
write_csv("summary_previous_vs_new.csv", ["measure", "previous", "new"], [[l, a, b] for l, a, b in rows])

                                    previous             new        change
TS valid px                    1,000,130,520   1,000,246,561      +116,041
TI valid px                    1,019,081,899   1,019,081,899            +0
both valid px                    996,647,176     996,757,493      +110,317
TS-only px                         3,483,344       3,489,068        +5,724
TI-only px                        22,434,723      22,324,406      -110,317
non-overlap px                    25,918,067      25,813,474      -104,593
agreeing px                      566,475,640     566,344,299      -131,341
TS code 212 px                    24,465,030      24,405,752       -59,278
TS codes with a 0 digit px            57,282               0       -57,282
TS Persistent px                 108,412,569     107,997,320      -415,249
TS Deformation px                 35,761,712      36,176,961      +415,249
TS Perforation px                 18,522,260      18,637,559      +115,299
agreement                

In [143]:
### Check: edge in the middle years where edge = 0 in 1990 and 2020 ###
NAMES = ["NoData", "Attrition", "Aggregation", "Shrinkage", "Perforation", "Deformation",
         "Persistent", "Enlargement", "Dissection", "Frag per se", "Creation"]
n_px  = np.zeros(11, np.int64)   # pixels per new TS class
n_mid = np.zeros(11, np.int64)   # ... with edge > 0 in at least one middle year
yrs_sum = np.zeros(11, np.int64) # total number of years with edge > 0
s = {"mask": rasterio.open(F["mask"]), "ts": rasterio.open(F["new_ts_rc"])}
yr = [rasterio.open(p) for p in MW["edge"]]
t0 = time.time()
try:
    windows = list(strips(s["ts"]))
    for i, w in enumerate(windows, 1):
        keep = read(s["mask"], w)[0] == 1
        e0, v0 = read(yr[0], w); e1, v1 = read(yr[-1], w)
        sel = keep & (~v0 | (e0 == 0)) & (~v1 | (e1 == 0))
        if sel.any():
            nyrs = np.zeros(sel.shape, np.int16)
            for src_ in yr[1:-1]:
                e, v = read(src_, w)
                nyrs += (v & (e > 0)).astype(np.int16)
            cls = read_class(s["ts"], w)[sel]
            n_px    += np.bincount(cls, minlength=11)
            n_mid   += np.bincount(cls[nyrs[sel] > 0], minlength=11)
            yrs_sum += np.bincount(cls, weights=nyrs[sel], minlength=11).astype(np.int64)
        if i % max(1, len(windows) // 10) == 0 or i == len(windows):
            print(f"  {i:,}/{len(windows):,} strips ({(time.time() - t0) / 60:.1f} min)")
finally:
    for x_ in [*s.values(), *yr]:
        x_.close()

print(f"{'new TS class':14}{'pixels':>14}{'edge>0 mid-series':>20}{'mean years >0':>15}")
for c in range(11):
    if n_px[c]:
        print(f"{NAMES[c]:14}{n_px[c]:>14,}{n_mid[c]:>14,} ({n_mid[c]/n_px[c]:5.1%}){yrs_sum[c]/n_px[c]:>15.2f}")
print(f"{'total':14}{n_px.sum():>14,}{n_mid.sum():>14,} ({n_mid.sum()/n_px.sum():5.1%})")


  75/755 strips (0.3 min)
  150/755 strips (1.0 min)
  225/755 strips (1.6 min)
  300/755 strips (4.1 min)
  375/755 strips (7.7 min)
  450/755 strips (10.9 min)
  525/755 strips (16.3 min)
  600/755 strips (22.5 min)
  675/755 strips (27.5 min)
  750/755 strips (29.3 min)
  755/755 strips (29.3 min)
new TS class          pixels   edge>0 mid-series  mean years >0
NoData                32,714        32,707 (100.0%)           9.26
Attrition                604           593 (98.2%)          11.89
Aggregation               58            56 (96.6%)          20.76
Shrinkage                 14            14 (100.0%)          21.29
Perforation           23,307        23,307 (100.0%)          10.13
Deformation          156,278       156,278 (100.0%)           8.70
Persistent         7,114,309       847,550 (11.9%)           0.58
Enlargement           44,669        44,661 (100.0%)          13.43
Frag per se               13            10 (76.9%)           9.15
Creation                  53       

In [145]:
### Edge: Python vs TerrSet, by number of years with edge = 0 ###
s = {"py": rasterio.open(Z["edge"]), "tr": rasterio.open(TERR["edge"]), "mask": rasterio.open(F["mask"])}
yr = [rasterio.open(p) for p in MW["edge"]]
GRP = (("0 zero years", 0, 0), ("1-5", 1, 5), ("6-15", 6, 15), ("16-30", 16, 30), ("all 31", 31, 31))
G = {g: Counter() for g, _, _ in GRP}
fp = Counter()
dig = lambda z: np.where(z > THR, 3, np.where(z < -THR, 1, 2))
t0 = time.time()
try:
    windows = list(strips(s["py"]))
    for i, w in enumerate(windows, 1):
        zp, vp = read(s["py"], w); zt, vt = read(s["tr"], w)
        zp, zt = zp.astype(np.float64), zt.astype(np.float64)
        keep = read(s["mask"], w)[0] == 1
        nz = np.zeros(zp.shape, np.int16)
        for src_ in yr:
            e, v = read(src_, w)
            nz += (~v | (e == 0)).astype(np.int16)
        po, to = vp & ~vt, ~vp & vt
        fp["Python-only"] += int(po.sum())
        fp["... edge 0 in all 31 years"] += int((po & (nz == 31)).sum())
        fp["Python valid with edge 0 in all 31 years"] += int((vp & (nz == 31)).sum())
        fp["... TerrSet NoData there"] += int((vp & (nz == 31) & ~vt).sum())
        fp["TerrSet-only"] += int(to.sum())
        fp["... inside the pn mask"] += int((to & keep).sum())
        b = vp & vt
        for g, lo, hi in GRP:
            sel = b & (nz >= lo) & (nz <= hi)
            if sel.any():
                x, y = zp[sel], zt[sel]; d = x - y; S = G[g]
                S["n"] += x.size; S["sx"] += x.sum(); S["sy"] += y.sum()
                S["sxx"] += (x * x).sum(); S["syy"] += (y * y).sum(); S["sxy"] += (x * y).sum()
                S["sd2"] += (d * d).sum(); S["max"] = max(S["max"], float(np.abs(d).max()))
                S["digit differs"] += int((dig(x) != dig(y)).sum())
        if i % max(1, len(windows) // 10) == 0 or i == len(windows):
            print(f"  {i:,}/{len(windows):,} strips ({(time.time() - t0) / 60:.1f} min)")
finally:
    for x_ in [*s.values(), *yr]:
        x_.close()

print("Footprint:")
for k, v in fp.items():
    print(f"  {k:45}{v:>15,}")
print(f"{'years with edge 0':18}{'pixels':>15}{'r':>10}{'RMSE':>9}{'max |diff|':>12}{'edge digit differs':>26}")
for g, _, _ in GRP:
    S = G[g]; n = S["n"]
    if not n:
        print(f"{g:18}{0:>15,}"); continue
    r = (n * S["sxy"] - S["sx"] * S["sy"]) / math.sqrt((n * S["sxx"] - S["sx"] ** 2) * (n * S["syy"] - S["sy"] ** 2))
    print(f"{g:18}{n:>15,}{r:>10.5f}{math.sqrt(S['sd2'] / n):>9.4f}{S['max']:>12.4f}"
          f"{S['digit differs']:>15,} ({S['digit differs'] / n:6.2%})")


  75/755 strips (5.4 min)
  150/755 strips (10.8 min)
  225/755 strips (16.4 min)
  300/755 strips (22.1 min)
  375/755 strips (28.1 min)
  450/755 strips (34.8 min)
  525/755 strips (42.6 min)
  600/755 strips (49.5 min)
  675/755 strips (55.9 min)
  750/755 strips (61.6 min)
  755/755 strips (61.9 min)
Footprint:
  Python-only                                        6,266,801
  ... edge 0 in all 31 years                         6,266,801
  Python valid with edge 0 in all 31 years           6,266,801
  ... TerrSet NoData there                           6,266,801
  TerrSet-only                                       9,261,441
  ... inside the pn mask                                     0
years with edge 0          pixels         r     RMSE  max |diff|        edge digit differs
0 zero years          930,585,554   0.99681   0.0807      0.2270     37,928,078 ( 4.08%)
1-5                    17,166,094   0.99696   0.0755      0.2237        599,010 ( 3.49%)
6-15                   28,984,064   

In [146]:
### Test: is the footprint mismatch caused by 212 + the stable band? ###
THRS = (0.5, 0.25, 0.1, 0.0)
RUNS = [f"+/-{t}" for t in THRS] + ["+/-0.5 + edge sign in 212"]
R = {k: Counter() for k in RUNS}
recon_bad = 0
ZB = (0.1, 0.25, 0.5)                                   # |z| bin edges
zbin = {m: np.zeros(4, np.int64) for m in ("edge", "pn")}
tau0 = Counter()

def dg(z, v, t):
    return np.where(v, np.where(z > t, 3, np.where(z < -t, 1, 2)), 0)

def acc(C, k, c212, tk, tv):
    kv = k > 0
    C["TS valid"] += int(kv.sum()); C["both"] += int((kv & tv).sum())
    C["TS-only"] += int((kv & ~tv).sum()); C["TI-only"] += int((~kv & tv).sum())
    C["agree"] += int((kv & tv & (k == tk)).sum())
    C["TS 212"] += int(c212.sum()); C["TI-only where TS 212"] += int((~kv & tv & c212).sum())

s = {m: rasterio.open(Z[m]) for m in METRICS}
s.update(ts=rasterio.open(F["new_ts_rc"]), ti=rasterio.open(F["ti_rc"]),
         te=rasterio.open(TAU["edge"]), tp=rasterio.open(TAU["pn"]))
t0 = time.time()
try:
    windows = list(strips(s["ts"]))
    for i, w in enumerate(windows, 1):
        z, v = {}, {}
        for m in METRICS:
            a, b = read(s[m], w); z[m], v[m] = a.astype(np.float64), b
        tk = read_class(s["ti"], w); tv = tk > 0
        pk = read_class(s["ts"], w)
        for t in THRS:
            code = dg(z["pn"], v["pn"], t) * 100 + dg(z["area"], v["area"], t) * 10 + dg(z["edge"], v["edge"], t)
            k = combined_class(code)
            c212 = code == 212
            acc(R[f"+/-{t}"], k, c212, tk, tv)
            if t == 0.5:
                recon_bad += int((k != pk).sum())
                k2 = k.copy()
                k2[c212 & (z["edge"] < 0)] = 3          # -> 211 Shrinkage
                k2[c212 & (z["edge"] > 0)] = 4          # -> 213 Perforation
                acc(R[RUNS[-1]], k2, c212 & (z["edge"] == 0), tk, tv)
                sel = (k == 0) & tv & c212              # TI-only pixels
                if sel.any():
                    for m in ("edge", "pn"):
                        zbin[m] += np.bincount(np.digitize(np.abs(z[m][sel]), ZB), minlength=4)
                    tau0["TI-only px"] += int(sel.sum())
                    tau0["edge Tau exactly 0"] += int((read(s["te"], w)[0][sel] == 0).sum())
                    tau0["pn Tau exactly 0"] += int((read(s["tp"], w)[0][sel] == 0).sum())
        if i % max(1, len(windows) // 10) == 0 or i == len(windows):
            print(f"  {i:,}/{len(windows):,} strips ({(time.time() - t0) / 60:.1f} min)")
finally:
    for x_ in s.values():
        x_.close()

print(f"Reconstruction at +/-0.5 vs saved combined_rc.tif: {recon_bad:,} pixels differ (should be ~0)")
print(f"{'TS rule':28}{'TS 212 km2':>12}{'TS-only km2':>13}{'TI-only km2':>13}{'TI-only = TS 212':>18}{'agreement':>11}")
for k in RUNS:
    C = R[k]
    print(f"{k:28}{C['TS 212'] * PX_KM2:>12,.0f}{C['TS-only'] * PX_KM2:>13,.0f}{C['TI-only'] * PX_KM2:>13,.0f}"
          f"{C['TI-only where TS 212'] * PX_KM2:>18,.0f}{100 * C['agree'] / max(1, C['both']):>10.2f}%")
print("(for the edge-sign row, 'TS 212' = 212 pixels with edge z exactly 0, still unclassified)")

n = tau0["TI-only px"]
print(f"TS z inside the TI-only pixels ({n:,} px):")
print(f"  {'':6}{'|z|<0.1':>14}{'0.1-0.25':>14}{'0.25-0.5':>14}{'>0.5':>10}")
for m in ("edge", "pn"):
    b = zbin[m]
    print(f"  {m:6}" + "".join(f"{x / n:>14.1%}" for x in b[:3]) + f"{b[3] / n:>10.1%}")
print(f"  edge Tau exactly 0 (no trend at all): {tau0['edge Tau exactly 0']:,} ({tau0['edge Tau exactly 0'] / n:.1%})")
print(f"  pn   Tau exactly 0 (no trend at all): {tau0['pn Tau exactly 0']:,} ({tau0['pn Tau exactly 0'] / n:.1%})")
                                                        

  75/755 strips (1.7 min)
  150/755 strips (3.9 min)
  225/755 strips (6.1 min)
  300/755 strips (8.3 min)
  375/755 strips (10.5 min)
  450/755 strips (13.0 min)
  525/755 strips (15.6 min)
  600/755 strips (18.2 min)
  675/755 strips (20.6 min)
  750/755 strips (22.6 min)
  755/755 strips (22.7 min)
Reconstruction at +/-0.5 vs saved combined_rc.tif: 0 pixels differ (should be ~0)
TS rule                       TS 212 km2  TS-only km2  TI-only km2  TI-only = TS 212  agreement
+/-0.5                            21,965        3,140       20,092            20,092     56.82%
+/-0.25                           10,900        3,253        9,140             9,140     64.12%
+/-0.1                             1,325        4,969        1,281             1,281     59.75%
+/-0.0                                 0        5,013            0                 0     60.86%
+/-0.5 + edge sign in 212              0        5,013            0                 0     56.19%
(for the edge-sign row, 'TS 212' = 212 

In [148]:
### Edge: is the Python Tau correct, and how does TerrSet compute it? (sample of windows) ###
rng = np.random.default_rng(0)
WS, NWIN = 256, 40
s = {"tau": rasterio.open(TAU["edge"]), "terr": rasterio.open(TERR["edge"])}
yr = [rasterio.open(p) for p in MW["edge"]]
print("edge raster dtype:", yr[0].dtypes[0])
H, W_ = s["tau"].height, s["tau"].width
cols = {k: [] for k in ("saved", "terr", "ta", "tb", "ties", "frac")}
tries = 0
try:
    while len(cols["saved"]) < NWIN and tries < 2000:
        tries += 1
        w = Window(int(rng.integers(0, W_ - WS)), int(rng.integers(0, H - WS)), WS, WS)
        t, tv = read(s["tau"], w); y, yv = read(s["terr"], w)
        b = tv & yv
        if b.sum() < 2000:
            continue
        RV = [read(src_, w) for src_ in yr]
        X = np.stack([d[b].astype(np.float64) for d, _ in RV])     # years x pixels
        X[~np.stack([v[b] for _, v in RV])] = np.nan
        n = np.isfinite(X).sum(0).astype(np.float64)
        S = np.zeros(X.shape[1]); T = np.zeros(X.shape[1])
        for i in range(len(yr) - 1):
            d = X[i + 1:] - X[i]
            S += np.nansum(np.sign(d), 0); T += (d == 0).sum(0)
        n0 = n * (n - 1) / 2
        cols["saved"].append(t[b].astype(np.float64)); cols["terr"].append(y[b].astype(np.float64))
        cols["ta"].append(S / n0); cols["tb"].append(S / np.sqrt(n0 * np.maximum(n0 - T, 1)))
        cols["ties"].append(T); cols["frac"].append(np.nanmax(np.abs(X - np.round(X)), 0))
finally:
    for x_ in [*s.values(), *yr]:
        x_.close()

C = {k: np.concatenate(v) for k, v in cols.items()}
r = lambda a, b: np.corrcoef(a, b)[0, 1]
print(f"{len(cols['saved'])} windows, {C['saved'].size:,} pixels valid in both")
print(f"Python saved Tau vs recomputed Tau: max |diff| = {np.abs(C['saved'] - C['ta']).max():.2e}  (should be ~1e-7)")
print(f"pixels with non-integer edge values: {(C['frac'] > 0).mean():.1%}")
print(f"{'':20}{'pixels':>12}{'r(TerrSet z, Tau)':>20}{'r(TerrSet z, Tau-b)':>22}")
for name, sel in (("all pixels", np.ones(C["ties"].size, bool)),
                  ("no tied years", C["ties"] == 0), ("with tied years", C["ties"] > 0)):
    if sel.sum() > 2:
        print(f"{name:20}{sel.sum():>12,}{r(C['terr'][sel], C['ta'][sel]):>20.5f}{r(C['terr'][sel], C['tb'][sel]):>22.5f}")


edge raster dtype: int32
40 windows, 2,181,263 pixels valid in both
Python saved Tau vs recomputed Tau: max |diff| = 2.97e-08  (should be ~1e-7)
pixels with non-integer edge values: 0.0%
                          pixels   r(TerrSet z, Tau)   r(TerrSet z, Tau-b)
all pixels             2,181,263             0.99711               0.99178
no tied years              7,270             0.99761               0.99761
with tied years        2,173,993             0.99710               0.99174


In [149]:
### Edge: does TerrSet match Tau with one year left out? ###
rng = np.random.default_rng(1)
WS, NWIN = 256, 10
s = {"tau": rasterio.open(TAU["edge"]), "terr": rasterio.open(TERR["edge"])}
yr = [rasterio.open(p) for p in MW["edge"]]
Xs, Ys = [], []
try:
    while len(Xs) < NWIN:
        w = Window(int(rng.integers(0, s["tau"].width - WS)), int(rng.integers(0, s["tau"].height - WS)), WS, WS)
        tv = read(s["tau"], w)[1]; y, yv = read(s["terr"], w)
        b = tv & yv
        if b.sum() < 2000:
            continue
        Xs.append(np.stack([read(src_, w)[0][b].astype(np.float64) for src_ in yr])); Ys.append(y[b].astype(np.float64))
finally:
    for x_ in [*s.values(), *yr]:
        x_.close()
X, Y = np.concatenate(Xs, 1), np.concatenate(Ys)

def tau(X):
    S = np.zeros(X.shape[1])
    for i in range(X.shape[0] - 1):
        S += np.sign(X[i + 1:] - X[i]).sum(0)
    n = X.shape[0]
    return S / (n * (n - 1) / 2)

r = lambda a: np.corrcoef(Y, a)[0, 1]
print(f"{Y.size:,} pixels | all 31 years: r = {r(tau(X)):.5f}")
res = sorted(((r(tau(np.delete(X, k, 0))), MW['edge'][k].name) for k in range(X.shape[0])), reverse=True)
for rv, name in res[:5]:
    print(f"  without {name:22} r = {rv:.5f}")


565,224 pixels | all 31 years: r = 0.99661
  without 2017_edge_1km.tif      r = 0.99825
  without 2018_edge_1km.tif      r = 0.99811
  without 2016_edge_1km.tif      r = 0.99802
  without 2019_edge_1km.tif      r = 0.99792
  without 2015_edge_1km.tif      r = 0.99768


In [150]:
### Class totals (km2) and % cover from the final typology rasters ###
import numpy as np, rasterio, time
from pathlib import Path
from rasterio.windows import Window

RASTERS = {
    "Time Interval 1990-2020": Path("./data/5_Typology/TimeInterval-1990-2020.tif"),
    "TS (new)":                Path("./data/5_Typology/combined_rc_timeseries/combined_rc.tif"),
}
NAMES = ["Attrition", "Aggregation", "Shrinkage", "Perforation", "Deformation",
         "Persistent", "Enlargement", "Dissection", "Fragmentation per se", "Creation"]
ROWS = 128

res = {}
for label, p in RASTERS.items():
    counts = np.zeros(256, np.int64)
    t0 = time.time()
    with rasterio.open(p) as src:
        px_km2 = abs(src.transform.a * src.transform.e) / 1e6     # pixel area from the raster itself
        for r in range(0, src.height, ROWS):
            w = Window(0, r, src.width, min(ROWS, src.height - r))
            a = src.read(1, window=w, masked=True)
            v = np.ma.getdata(a)[~np.ma.getmaskarray(a)].astype(np.int64)
            counts += np.bincount(v[(v >= 0) & (v < 256)], minlength=256)
    other = int(counts.sum() - counts[1:11].sum())
    res[label] = (counts[1:11] * px_km2, px_km2)
    print(f"{label}: pixel = {px_km2 * 1e6:.1f} m2, read in {(time.time() - t0) / 60:.1f} min"
          + (f" | WARNING {other:,} px with values outside 1-10" if other else ""))

labels = list(res)
print(f"{'Class':22}" + "".join(f"{l[:20]:>22}{'%cover':>10}" for l in labels))
for k, name in enumerate(NAMES):
    print(f"{name:22}" + "".join(f"{res[l][0][k]:>22,.1f}{100 * res[l][0][k] / res[l][0].sum():>10.4f}" for l in labels))
print(f"{'TOTAL':22}" + "".join(f"{res[l][0].sum():>22,.1f}{100:>10.4f}" for l in labels))


Time Interval 1990-2020: pixel = 1016.8 m2, read in 0.6 min
TS (new): pixel = 1016.8 m2, read in 0.7 min
Class                   Time Interval 1990-2    %cover              TS (new)    %cover
Attrition                          189,300.7   18.2690             140,142.3   13.7795
Aggregation                        142,334.1   13.7363             180,238.6   17.7220
Shrinkage                           74,800.6    7.2188              71,675.5    7.0475
Perforation                         30,873.1    2.9795              18,950.4    1.8633
Deformation                          2,330.2    0.2249              36,784.2    3.6168
Persistent                          26,344.8    2.5425             109,810.0   10.7971
Enlargement                        114,227.3   11.0238             109,914.9   10.8074
Dissection                         191,161.5   18.4486             109,974.3   10.8132
Fragmentation per se                 1,380.5    0.1332              66,854.9    6.5735
Creation                 

In [153]:
### Class totals (km2) and % cover from the final typology rasters ###
import numpy as np, rasterio, time
from pathlib import Path
from rasterio.windows import Window

RASTERS = {
    "Time Interval 1990-2020": Path("./data/5_Typology/TimeInterval-1990-2020.tif"),
    "TS (new)":                Path("./data/5_Typology/combined_rc_timeseries/combined_rc.tif"),
}
NAMES = ["Attrition", "Aggregation", "Shrinkage", "Perforation", "Deformation",
         "Persistent", "Enlargement", "Dissection", "Fragmentation per se", "Creation"]
ROWS = 128

res = {}
for label, p in RASTERS.items():
    counts = np.zeros(256, np.int64)
    t0 = time.time()
    with rasterio.open(p) as src:
        px_km2 = abs(src.transform.a * src.transform.e) / 1e6     # pixel area from the raster itself
        for r in range(0, src.height, ROWS):
            w = Window(0, r, src.width, min(ROWS, src.height - r))
            a = src.read(1, window=w, masked=True)
            v = np.ma.getdata(a)[~np.ma.getmaskarray(a)].astype(np.int64)
            counts += np.bincount(v[(v >= 0) & (v < 256)], minlength=256)
    other = int(counts.sum() - counts[1:11].sum())
    res[label] = (counts[1:11] * px_km2, px_km2)
    print(f"{label}: pixel = {px_km2 * 1e6:.1f} m2, read in {(time.time() - t0) / 60:.1f} min"
          + (f" | WARNING {other:,} px with values outside 1-10" if other else ""))

labels = list(res)
print(f"{'Class':22}" + "".join(f"{l[:20]:>22}{'%cover':>10}" for l in labels))
for k, name in enumerate(NAMES):
    print(f"{name:22}" + "".join(f"{res[l][0][k]:>22,.1f}{100 * res[l][0][k] / res[l][0].sum():>10.4f}" for l in labels))
    print(f"{'TOTAL':22}" + "".join(f"{res[l][0].sum():>22,.1f}{100:>10.4f}" for l in labels))


Time Interval 1990-2020: pixel = 1016.8 m2, read in 0.7 min
TS (new): pixel = 1016.8 m2, read in 0.6 min
Class                   Time Interval 1990-2    %cover              TS (new)    %cover
Attrition                          189,300.7   18.2690             140,142.3   13.7795
TOTAL                            1,036,186.4  100.0000           1,017,034.9  100.0000
Aggregation                        142,334.1   13.7363             180,238.6   17.7220
TOTAL                            1,036,186.4  100.0000           1,017,034.9  100.0000
Shrinkage                           74,800.6    7.2188              71,675.5    7.0475
TOTAL                            1,036,186.4  100.0000           1,017,034.9  100.0000
Perforation                         30,873.1    2.9795              18,950.4    1.8633
TOTAL                            1,036,186.4  100.0000           1,017,034.9  100.0000
Deformation                          2,330.2    0.2249              36,784.2    3.6168
TOTAL                    

In [159]:
for label, p in RASTERS.items():
    counts = np.zeros(256, np.int64)
    with rasterio.open(p) as src:
        px_m2 = abs(src.transform.a * src.transform.e)
        for r in range(0, src.height, ROWS):
            w = Window(0, r, src.width, min(ROWS, src.height - r))
            a = src.read(1, window=w, masked=True)
            v = np.ma.getdata(a)[~np.ma.getmaskarray(a)].astype(np.int64)
            counts += np.bincount(v[(v >= 0) & (v < 256)], minlength=256)
    c = counts[1:11]
    print(f"{label}  (raster pixel = {px_m2:.1f} m2)")
    print(f"{'Class':22}{'pixel count':>16}{'km2 (30 m px)':>16}{'km2 (raster px)':>18}{'%cover':>10}")
    for k, name in enumerate(NAMES):
        print(f"{name:22}{c[k]:>16,}{c[k] * 0.0009:>16,.1f}{c[k] * px_m2 / 1e6:>18,.1f}{100 * c[k] / c.sum():>10.4f}")
    print(f"{'TOTAL':22}{c.sum():>16,}{c.sum() * 0.0009:>16,.1f}{c.sum() * px_m2 / 1e6:>18,.1f}{100:>10.4f}")

Time Interval 1990-2020  (raster pixel = 1016.8 m2)
Class                      pixel count   km2 (30 m px)   km2 (raster px)    %cover
Attrition                  186,175,860       167,558.3         189,300.7   18.2690
Aggregation                139,984,600       125,986.1         142,334.1   13.7363
Shrinkage                   73,565,852        66,209.3          74,800.6    7.2188
Perforation                 30,363,483        27,327.1          30,873.1    2.9795
Deformation                  2,291,771         2,062.6           2,330.2    0.2249
Persistent                  25,909,886        23,318.9          26,344.8    2.5425
Enlargement                112,341,729       101,107.6         114,227.3   11.0238
Dissection                 188,005,937       169,205.3         191,161.5   18.4486
Fragmentation per se         1,357,667         1,221.9           1,380.5    0.1332
Creation                   259,085,114       233,176.6         263,433.7   25.4234
TOTAL                    1,019,081,